# E-commerce Inventory Optimization & Reorder Decision Analysis

This project analyzes e-commerce transaction data to identify sales performance, inventory health, slow-moving SKUs, and reorder recommendations.

**Business Goal:** Move beyond basic reporting and create a simple decision-support workflow for inventory planning.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Load Data

The dataset is loaded from an Excel file. If you use a CSV file instead, replace `pd.read_excel()` with `pd.read_csv()`.

In [ ]:
# Load the transaction dataset
df = pd.read_excel("Online Retail.xlsx")

# Preview the dataset
df.head()

## 3. Data Cleaning

Real-world sales data often includes cancellations, returns, missing product names, or invalid prices. This section cleans the dataset before analysis.

In [ ]:
# Standardize column names
df.columns = df.columns.str.strip().str.lower()

# Rename columns for easier business interpretation
df = df.rename(columns={
    "stockcode": "sku",
    "description": "product_name",
    "invoicedate": "invoice_date",
    "unitprice": "unit_price"
})

# Convert invoice date to datetime if available
if "invoice_date" in df.columns:
    df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")

# Keep valid sales only: positive quantity and positive price
df = df[(df["quantity"] > 0) & (df["unit_price"] > 0)].copy()

# Remove rows with missing SKU or product name
df = df.dropna(subset=["sku", "product_name"])

# Create revenue column
df["revenue"] = df["quantity"] * df["unit_price"]

# Check cleaned data
df.head()

## 4. Product Category Creation

Because the dataset does not include clean product categories, this project creates simple rule-based categories from product names. In a real company, this could be replaced with a product master table.

In [ ]:
def categorize(desc):

    desc = str(desc).lower()

    # Home decor (largest category in this dataset)
    if any(word in desc for word in [
        "candle", "holder", "lamp", "frame",
        "mirror", "vase", "clock", "heart",
        "flower", "light", "decor", "ornament",
        "door", "picture", "sign", "hanger",
        "shelf", "storage", "box", "drawer"
    ]):
        return "home decor"

    # Kitchen / dining
    elif any(word in desc for word in [
        "kitchen", "tea", "coffee", "mug",
        "cup", "plate", "bowl", "glass",
        "bottle", "jar", "tin", "cutlery",
        "spoon", "fork", "knife", "tray",
        "baking", "pan", "kettle", "teapot",
        "cake"
    ]):
        return "kitchen"

    # Bags
    elif any(word in desc for word in [
        "bag", "purse", "tote",
        "lunch bag", "shopping bag",
        "backpack"
    ]):
        return "bags"

    # Gift / decorative
    elif any(word in desc for word in [
        "gift", "present", "ribbon",
        "card", "wrap", "wrapping",
        "party", "bunting", "decorative"
    ]):
        return "gift/decorative"

    # Seasonal
    elif any(word in desc for word in [
        "christmas", "xmas", "santa",
        "snow", "tree", "easter",
        "halloween", "valentine",
        "seasonal"
    ]):
        return "seasonal"

    # Sets / bundles
    elif any(word in desc for word in [
        "set", "kit", "pack", "bundle"
    ]):
        return "sets"

    # Stationery
    elif any(word in desc for word in [
        "paper", "notebook", "note",
        "stationery", "pen", "pencil",
        "file", "folder", "sticker"
    ]):
        return "stationery"

    else:
        return "other"


# Apply category mapping
df["category"] = df["product_name"].apply(categorize)

df[["sku", "product_name", "category"]].head()

## 5. Sales Trend Analysis

This section reviews overall monthly sales performance to understand demand patterns over time.

In [ ]:
# Monthly sales trend
if "invoice_date" in df.columns:
    monthly_sales = (
        df.set_index("invoice_date")
          .resample("ME")
          .agg(units_sold=("quantity", "sum"), revenue=("revenue", "sum"))
          .reset_index()
    )

    display(monthly_sales.head())

    plt.figure(figsize=(10, 5))
    plt.plot(monthly_sales["invoice_date"], monthly_sales["revenue"], marker="o")
    plt.title("Monthly Revenue Trend")
    plt.xlabel("Month")
    plt.ylabel("Revenue")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("invoice_date column is not available, so monthly trend analysis is skipped.")

## 6. Best Seller & Category Performance

This section identifies top-selling SKUs and category-level performance.

In [ ]:
# SKU-level sales performance
sku_sales = (
    df.groupby("sku", as_index=False)
      .agg(
          product_name=("product_name", "first"),
          category=("category", "first"),
          units_sold=("quantity", "sum"),
          revenue=("revenue", "sum"),
          avg_unit_price=("unit_price", "mean")
      )
)

# Best sellers by units sold
top_skus = sku_sales.sort_values("units_sold", ascending=False).head(10)
display(top_skus)

plt.figure(figsize=(10, 5))
plt.bar(top_skus["sku"].astype(str), top_skus["units_sold"])
plt.title("Top 10 Best-Selling SKUs by Units Sold")
plt.xlabel("SKU")
plt.ylabel("Units Sold")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Category-level performance
# Category summary excluding "other" for clearer business interpretation

category_summary = (
    df[df["category"] != "other"]
    .groupby("category")
    .agg(
        total_units_sold=("quantity", "sum"),
        total_revenue=("revenue", "sum"),
        sku_count=("sku", "nunique")
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

category_summary

plt.figure(figsize=(10, 5))

plt.bar(
    category_summary["category"],
    category_summary["total_revenue"]
)

plt.title("Revenue by Product Category (Excluding Other)")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Inventory Health Analysis

The original dataset does not include real beginning or ending inventory. To demonstrate inventory decision logic, this project simulates inventory based on sales volume.

In a real business setting, this section should use actual on-hand inventory from a WMS, ERP, or Shopify inventory export.

In [ ]:
# Simulate inventory for portfolio demonstration
np.random.seed(42)

inventory = sku_sales.copy()
inventory["begin_inventory"] = inventory["units_sold"] + np.random.randint(10, 100, size=len(inventory))
inventory["ending_inventory"] = inventory["begin_inventory"] - inventory["units_sold"]

# Inventory KPIs
inventory["sell_through_rate"] = inventory["units_sold"] / inventory["begin_inventory"]
inventory["avg_inventory"] = (inventory["begin_inventory"] + inventory["ending_inventory"]) / 2
inventory["inventory_turnover"] = inventory["units_sold"] / inventory["avg_inventory"]

# Inventory status flags
inventory["overstock_flag"] = inventory["ending_inventory"] > inventory["units_sold"]
inventory["stockout_risk_flag"] = inventory["ending_inventory"] < inventory["units_sold"] * 0.20

inventory.head()

In [ ]:
# Inventory KPI summary
inventory_summary = pd.DataFrame({
    "metric": [
        "Total SKUs",
        "Average Sell-Through Rate",
        "Average Inventory Turnover",
        "Overstock SKUs",
        "Stockout Risk SKUs"
    ],
    "value": [
        inventory["sku"].nunique(),
        inventory["sell_through_rate"].mean(),
        inventory["inventory_turnover"].mean(),
        inventory["overstock_flag"].sum(),
        inventory["stockout_risk_flag"].sum()
    ]
})

display(inventory_summary)

## 8. Slow-Moving SKU Detection

Slow-moving SKUs are items with low sell-through and remaining inventory. These products may create overstock risk or tie up cash in inventory.

In [ ]:
# Define slow-moving SKUs
slow_moving_skus = (
    inventory[
        (inventory["sell_through_rate"] < 0.30) &
        (inventory["ending_inventory"] > 0)
    ]
    .sort_values(["sell_through_rate", "ending_inventory"], ascending=[True, False])
)

display(slow_moving_skus[[
    "sku", "product_name", "category", "units_sold", "ending_inventory",
    "sell_through_rate", "inventory_turnover"
]].head(15))

## 9. Reorder Recommendation Logic

This section turns the analysis into a business recommendation. The logic estimates reorder needs using average demand, lead time, and safety stock.

**Formula used:**

`Reorder Point = Expected Demand During Lead Time + Safety Stock`

For this portfolio project, weekly demand is estimated from historical sales.

In [ ]:
# Estimate demand assumptions
if "invoice_date" in df.columns:
    date_range_days = (df["invoice_date"].max() - df["invoice_date"].min()).days
    weeks_in_data = max(date_range_days / 7, 1)
else:
    weeks_in_data = 52

lead_time_weeks = 4
safety_stock_weeks = 2

inventory["estimated_weekly_demand"] = inventory["units_sold"] / weeks_in_data
inventory["expected_demand_lead_time"] = inventory["estimated_weekly_demand"] * lead_time_weeks
inventory["safety_stock"] = inventory["estimated_weekly_demand"] * safety_stock_weeks
inventory["reorder_point"] = inventory["expected_demand_lead_time"] + inventory["safety_stock"]

inventory["recommended_reorder_qty"] = (
    inventory["reorder_point"] - inventory["ending_inventory"]
).clip(lower=0).round(0)

# Create business recommendation label
def make_recommendation(row):
    if row["recommended_reorder_qty"] > 0 and row["stockout_risk_flag"]:
        return "Reorder immediately"
    elif row["recommended_reorder_qty"] > 0:
        return "Monitor and reorder soon"
    elif row["overstock_flag"] or row["sell_through_rate"] < 0.30:
        return "Consider promotion or bundle"
    else:
        return "Healthy inventory"

inventory["recommendation"] = inventory.apply(make_recommendation, axis=1)

# Review recommendation output
recommendation_table = inventory[[
    "sku", "product_name", "category", "units_sold", "ending_inventory",
    "sell_through_rate", "estimated_weekly_demand", "reorder_point",
    "recommended_reorder_qty", "recommendation"
]].sort_values(["recommended_reorder_qty", "units_sold"], ascending=[False, False])

display(recommendation_table.head(20))

## 10. Business Insights & Action Plan

This section summarizes the analysis in a decision-ready format. Instead of stopping at charts, the project explains what actions the business should take.

In [ ]:
# Create insight summary
total_skus = inventory["sku"].nunique()
stockout_risk_count = inventory["stockout_risk_flag"].sum()
overstock_count = inventory["overstock_flag"].sum()
slow_moving_count = slow_moving_skus["sku"].nunique()
reorder_count = (inventory["recommended_reorder_qty"] > 0).sum()

insights = pd.DataFrame({
    "business_question": [
        "How many SKUs may need reorder action?",
        "How many SKUs show stockout risk?",
        "How many SKUs show overstock risk?",
        "How many SKUs are slow-moving?"
    ],
    "finding": [
        f"{reorder_count} out of {total_skus} SKUs have a positive recommended reorder quantity.",
        f"{stockout_risk_count} SKUs have low ending inventory compared with sales velocity.",
        f"{overstock_count} SKUs have ending inventory higher than units sold.",
        f"{slow_moving_count} SKUs have low sell-through and remaining inventory."
    ],
    "recommended_action": [
        "Prioritize reorder review for high-demand SKUs with low ending inventory.",
        "Prevent lost sales by monitoring fast-moving low-stock items.",
        "Use bundles, markdowns, or reduced future buys to manage excess inventory.",
        "Review product placement, pricing, and future purchasing quantities."
    ]
})

display(insights)

In [ ]:
# Save final recommendation table for portfolio output
recommendation_table.to_csv("inventory_reorder_recommendations.csv", index=False)

print("Saved: inventory_reorder_recommendations.csv")

## 11. Portfolio Summary

This project demonstrates how sales and inventory data can be used to support business decisions. The analysis identifies best sellers, category performance, inventory health, slow-moving products, and reorder opportunities.

**Key portfolio takeaway:** This is not only a reporting project. It shows how analytics can support inventory planning decisions such as when to reorder, which SKUs need monitoring, and which products may need promotion or bundling.

## 6. AI Inventory Decision Assistant

In this section, we simulate a simple **AI-assisted inventory decision system** that recommends actions based on inventory health indicators.

**Decision Rules**
- **Stockout Risk** → Reorder Immediately  
- **Overstock** → Run Promotion / Bundle  
- **Healthy Inventory** → Maintain Inventory


In [ ]:
# AI Inventory Decision Assistant
# Generate inventory recommendations based on stock health

def inventory_ai_assistant(row):
    if row["stockout_risk_flag"]:
        return "Reorder Immediately"
    elif row["overstock_flag"]:
        return "Run Promotion / Bundle"
    else:
        return "Maintain Inventory"


# Apply recommendation logic to the existing inventory dataframe
inventory["ai_recommendation"] = inventory.apply(
    inventory_ai_assistant,
    axis=1
)

# Recommendation table
ai_recommendation_table = inventory[[
    "sku",
    "product_name",
    "category",
    "units_sold",
    "ending_inventory",
    "stockout_risk_flag",
    "overstock_flag",
    "ai_recommendation"
]].head(10)

print("AI Inventory Recommendations")
ai_recommendation_table


## 7. Business Insights & Action Plan

### Key Findings
- Products with **stockout risk** may require immediate replenishment to avoid lost sales.
- **Overstocked SKUs** may benefit from promotions, bundling, or markdown strategies.
- Stable-performing products should maintain healthy inventory levels.

### Recommended Actions
1. Reorder high-demand products with low inventory.
2. Run promotions or bundles for slow-moving SKUs.
3. Monitor sell-through and inventory turnover regularly for better inventory planning.


## 12. Export CSVs for Tableau Dashboard

Export analysis results to CSV files for Tableau visualization.

In [ ]:
import os
os.makedirs("../outputs", exist_ok=True)

# Monthly revenue trend
monthly_sales.rename(columns={"invoice_date": "month"}, inplace=True)
monthly_sales["month"] = monthly_sales["month"].dt.strftime("%Y-%m")
monthly_sales.to_csv("../outputs/monthly_sales.csv", index=False)

# Category performance (excluding 'other')
category_summary["revenue_pct"] = (
    category_summary["total_revenue"] / category_summary["total_revenue"].sum() * 100
).round(2)
category_summary.to_csv("../outputs/category_performance.csv", index=False)

# Slow-moving SKUs
slow_moving_skus[[
    "sku", "product_name", "category", "units_sold",
    "ending_inventory", "sell_through_rate", "inventory_turnover"
]].to_csv("../outputs/slow_moving_skus.csv", index=False)

# Full reorder recommendation table
recommendation_table.to_csv("../outputs/inventory_reorder_recommendations.csv", index=False)

print("Saved:")
print("  ../outputs/monthly_sales.csv")
print("  ../outputs/category_performance.csv")
print("  ../outputs/slow_moving_skus.csv")
print("  ../outputs/inventory_reorder_recommendations.csv")